In [5]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline

from scikeras.wrappers import KerasClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [7]:
# load dataset
data = pd.read_csv("Churn_Modelling.csv")

# drop unnecessary columns
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

# label encode gender
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

# one hot encode geography
ohe = OneHotEncoder(handle_unknown='ignore')
geo_encoded = ohe.fit_transform(data[['Geography']]).toarray()

geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(['Geography'])
)

# concat encoded columns and drop original
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

# split into X and y
X = data.drop('Exited', axis=1)
y = data['Exited']


#split train test
X_train,X_test,y_train, y_test=train_test_split(X,y,test_size=0.2,random_state=42)

#scale
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

#save encoders and scalers
with open('label_encoder_gender.pkl','wb') as file:
  pickle.dump(label_encoder_gender, file)

with open('ohe.pkl', 'wb') as file:
  pickle.dump(ohe, file)

with open('scaler.pkl', 'wb') as file:
  pickle.dump(scaler, file)


In [8]:
#define a function to create the model and try different parameters(Keras Classifier)

def create_model(neurons=32, layers=1):
  model=Sequential()
  model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))

  for _ in range(layers-1):
    model.add(Dense(neurons, activation='relu'))

  model.add(Dense(1,activation='sigmoid'))
  model.compile(optimizer='adam', loss='binary_crossentropy',metrics=['accuracy'])

  return model



In [12]:
#create a keras classifier
model=KerasClassifier(layers=1, neurons=32,build_fn=create_model, epochs=50, batch_size=10, verbose=0)


In [13]:
#define grid search parameters
param_grid={
  'neurons':[16,32,64,128],
  'layers':[1,2],
  'epochs':[50,100]
}

In [14]:
#perform grid search cv
grid=GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3)
grid_result=grid.fit(X_train, y_train)

#print the best params
print("Best: %f using %s" % (grid_result.best_score_,grid_result.best_params_))

d:\conda_envs\DL1\Lib\site-packages\scikeras\wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)





Best: 0.858250 using {'epochs': 50, 'layers': 1, 'neurons': 128}
